# 2. ETF Screening and Selection

The data collection chapter gave us the raw universe. This chapter applies a
sequence of filters to remove ETFs that are too small, too expensive, poorly
tracked, or simply unavailable on a zero-fee UK ISA broker. Whatever survives
is ranked by risk-adjusted return — and only the top-ranked fund(s) per region
make the shortlist.

The goal is a **small, high-conviction list**, not exhaustive coverage. Fewer
lines in the portfolio means lower operational overhead at each annual rebalance.

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("..").resolve()))

import os
import json
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
import requests
from scipy import stats

from etf_utils.config import DATA_RAW, DATA_INTERMEDIATE, DATA_OUTPUT, DATA_CONFIG
from etf_utils.data_provider import DataProvider
from etf_utils.data_io import get_region_category_from_filename, get_asset_class_from_filename
from etf_utils.metrics import calculate_annualized_volatility
from etf_utils.platform_check import check_platform
from etf_utils.database import save_screened_etfs, save_benchmark_etfs

provider = DataProvider()


## Benchmarks for each asset class

Every screening filter that involves performance (the [beta](../content/99_glossary.md#risk-metrics)
check, the [Sharpe ratio](../content/99_glossary.md#risk-metrics) ranking)
needs a reference index to compare against. The table below shows the
benchmark chosen for each asset class and why.

| Asset class | Benchmark ticker | Why this benchmark |
|---|---|---|
| Equities | VEVE.L | Broad developed-market index; the most widely held global equity ETF by UK investors |
| Bonds | SAAA.L | Short-duration investment-grade aggregate; liquid, low-volatility reference |
| Precious metals | SGLN.L | Physical gold ETC; the dominant component in any mixed-metal basket |
| Commodities | CMOP.L | Broad Bloomberg Commodity index; covers energy, agriculture and metals in one ticker |

Benchmark returns and volatility are pulled for the same period used in the
[portfolio construction](03_portfolio_construction.ipynb) step, so the two
chapters are self-consistent.

In [2]:
provider.provider


'alphavantage'

In [3]:
# Calculate benchmark returns and volatility for 2025
benchmark_year_start = "2025-01-01"
benchmark_year_end = "2025-12-31"

eq_benchmark_symbol  = "VEVE"   # VEVE.L on yfinance
bnd_benchmark_symbol = "SAAA"   # SAAA.L on yfinance
pm_benchmark_symbol  = "SGLN"   # iShares Physical Gold ETC
cmd_benchmark_symbol = "CMOP"   # Invesco Bloomberg Commodity ETC

def _period_metrics(symbol, start, end):
    ret  = round(provider.get_benchmark_period_return(symbol, start, end) * 100, 2)
    yr = provider.get_historical_prices(symbol, start_date=start, end_date=end)["close"]
    vol  = round(calculate_annualized_volatility(yr) * 100, 2)
    sr   = round(ret / vol, 2) if vol > 0 else 0
    return ret, vol, sr

eq_benchmark_return,  eq_benchmark_volatility,  eq_sharpe_ratio  = _period_metrics(eq_benchmark_symbol,  benchmark_year_start, benchmark_year_end)
bnd_benchmark_return, bnd_benchmark_volatility, bond_sharpe_ratio = _period_metrics(bnd_benchmark_symbol, benchmark_year_start, benchmark_year_end)
pm_benchmark_return,  pm_benchmark_volatility,  pm_sharpe_ratio  = _period_metrics(pm_benchmark_symbol,  benchmark_year_start, benchmark_year_end)
cmd_benchmark_return, cmd_benchmark_volatility, cmd_sharpe_ratio = _period_metrics(cmd_benchmark_symbol, benchmark_year_start, benchmark_year_end)

print(f"2025 VEVE  return: {eq_benchmark_return}%,  vol: {eq_benchmark_volatility}%,  Sharpe: {eq_sharpe_ratio}")
print(f"2025 SAAA  return: {bnd_benchmark_return}%, vol: {bnd_benchmark_volatility}%, Sharpe: {bond_sharpe_ratio}")
print(f"2025 SGLN  return: {pm_benchmark_return}%,  vol: {pm_benchmark_volatility}%,  Sharpe: {pm_sharpe_ratio}")
print(f"2025 CMOP  return: {cmd_benchmark_return}%, vol: {cmd_benchmark_volatility}%, Sharpe: {cmd_sharpe_ratio}")


2025 VEVE  return: 13.81%,  vol: 14.71%,  Sharpe: 0.94
2025 SAAA  return: 2.96%, vol: 4.94%, Sharpe: 0.6
2025 SGLN  return: 53.66%,  vol: 17.65%,  Sharpe: 3.04
2025 CMOP  return: 8.23%, vol: 13.46%, Sharpe: 0.61


In [4]:
benchmark_etfs = [
    # Structure: [Asset Class, Region, Country, Primary Ticker, Description]
    
    # Equity Benchmarks - Developed Markets
    ['Equity', 'Developed_AmericasandUK', 'United Kingdom', 'ISF.L', 'iShares Core FTSE 100 UCITS ETF GBP (Dist)'],
    ['Equity', 'Developed_AmericasandUK', 'United States', 'SPY', 'SPDR S&P 500 ETF Trust'],
    ['Equity', 'Developed_AmericasandUK', 'Canada', 'ZCN.TO', 'BMO S&P/TSX Capped Composite'],    
    ['Equity', 'Developed_EMEA', 'Germany', 'CS51.L', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'France', 'CS51.L', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Italy', 'CS51.L', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Spain', 'CS51.L', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Netherlands', 'CS51.L', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Switzerland', 'CSWG.L', 'Amundi Index Solutions - Amundi MSCI Switzerland'],    
    ['Equity', 'Developed_APAC', 'Japan', 'XDJP.L', 'Xtrackers Nikkei 225 UCITS ETF 1D'],
    ['Equity', 'Developed_APAC', 'Australia', 'SAUS.L', 'iShares MSCI Australia UCITS ETF'],    
    # Equity Benchmarks - Emerging Markets
    ['Equity', 'Emerging_Americas', 'Brazil', 'IBZL.L', 'iShares MSCI Brazil UCITS ETF'],
    ['Equity', 'Emerging_Americas', 'Mexico', 'XMEX.L', 'iShares MSCI Mexico Capped ETF'],    
    ['Equity', 'Emerging_APACandEMEA', 'China', 'ASHR', 'XTRACKERS HARVEST CSI 300 CHINA A-SHARES ETF'],
    ['Equity', 'Emerging_APACandEMEA', 'India', 'XNIF.L', 'Xtrackers Nifty 50 Swap UCITS ETF 1C'],
    ['Equity', 'Emerging_APACandEMEA', 'South Korea', 'EWY', 'iShares MSCI South Korea ETF'],  
    ['Equity', 'Emerging_APACandEMEA', 'Indonesia', 'EIDO', 'iShares MSCI Indonesia UCITS ETF']
]

benchmark_df = pd.DataFrame(
    benchmark_etfs, 
    columns=['asset_class', 'region', 'country', 'benchmark_ticker', 'benchmark_description']
)

# Sort the DataFrame
benchmark_df = benchmark_df.sort_values(['asset_class', 'region', 'country'])

# Calculate 2025 returns for each benchmark ETF using DataProvider
benchmark_df['benchmark_2025_Return'] = benchmark_df['benchmark_ticker'].apply(
    lambda sym: round(provider.get_benchmark_period_return(sym, "2025-01-01", "2025-12-31") * 100, 2)
)
benchmark_df


,asset_class,region,country,benchmark_ticker,benchmark_description,benchmark_2025_Return
10,Equity,Developed_APAC,Australia,SAUS.L,iShares MSCI Australia UCITS ETF,6.23
9,Equity,Developed_APAC,Japan,XDJP.L,Xtrackers Nikkei 225 UCITS ETF 1D,19.26
2,Equity,Developed_AmericasandUK,Canada,ZCN.TO,BMO S&P/TSX Capped Composite,31.51
0,Equity,Developed_AmericasandUK,United Kingdom,ISF.L,iShares Core FTSE 100 UCITS ETF GBP (Dist),21.93
1,Equity,Developed_AmericasandUK,United States,SPY,SPDR S&P 500 ETF Trust,17.72
4,Equity,Developed_EMEA,France,CS51.L,iShares VII PLC - iShares Core EURO STOXX 50 E...,28.21
3,Equity,Developed_EMEA,Germany,CS51.L,iShares VII PLC - iShares Core EURO STOXX 50 E...,28.21
5,Equity,Developed_EMEA,Italy,CS51.L,iShares VII PLC - iShares Core EURO STOXX 50 E...,28.21
7,Equity,Developed_EMEA,Netherlands,CS51.L,iShares VII PLC - iShares Core EURO STOXX 50 E...,28.21
6,Equity,Developed_EMEA,Spain,CS51.L,iShares VII PLC - iShares Core EURO STOXX 50 E...,28.21


## Screening filters applied per asset class

A fund only survives if it clears **every** filter for its asset class. The
filters are cumulative — applied in order from broadest to most specific.

### Equities & Bonds
- **[Distributing](../content/99_glossary.md#wrappers-product-types) only** — dividends arrive as
  cash in the ISA account, ready to fund the next annual rebalance.
- **Size ≥ £100M** — smaller funds carry closure and wide-spread risk.
- **[TER](../content/99_glossary.md#costs) < 0.50%** — keeps the annual cost drag well below the
  long-run equity risk premium.
- **Available on InvestEngine or Trading212** — must be buyable at zero
  commission inside a UK ISA.

### Precious Metals
- Same size and platform filters as equities.
- **[TER](../content/99_glossary.md#costs) < 0.60%** — physical storage costs justify a slightly
  higher threshold.
- ETCs that are currency-hedged are excluded (hedging costs and tracking
  error outweigh the FX protection for long-horizon holders).

### Commodities
- Same size, TER and platform filters.
- **[Beta](../content/99_glossary.md#risk-metrics) ≥ 1.0 vs CMOP** — ensures the selected ETC
  actually tracks the broad commodity index, not a narrow sub-sector.
- Pure energy or agriculture ETCs are excluded; only broad diversified ETCs
  (energy + agriculture + metals in one wrapper) are retained.

ETFs that clear the filters are then **ranked by composite
[Sharpe ratio](../content/99_glossary.md#risk-metrics)** (50% 1-yr, 30% 3-yr, 20% 5-yr).
The top-ranked fund per region enters the portfolio shortlist.

In [5]:
main_df_equities = pd.DataFrame()

for filepath in sorted(DATA_RAW.glob("justetf_class-equity*.csv")):
    csvl = filepath.name
    try:
        asset_class = get_asset_class_from_filename(csvl)
        region_category = get_region_category_from_filename(csvl)

        if not asset_class or region_category == 'Unknown' or asset_class != 'equity':
            print(f'Skipping {csvl}')
            continue

        print(f'Processing {asset_class} file for {region_category}: {csvl}')

        try:
            csv_df = pd.read_csv(filepath)
            if csv_df.empty:
                print(f'Empty file: {csvl}')
                continue
        except pd.errors.EmptyDataError:
            print(f'Empty file: {csvl}')
            continue

        csv_df['asset_class'] = asset_class
        csv_df['region'] = csv_df['region'].fillna('N/A')
        csv_df['country'] = csv_df['country'].fillna('N/A')
        csv_df['platform'] = csv_df['ticker'].apply(check_platform)
        csv_df = csv_df[csv_df['platform'].notna()]

        distributing_df = csv_df.copy()
        distributing_df = distributing_df[distributing_df['dividends'] == 'Distributing']
        distributing_df = distributing_df[distributing_df['size'] > 100]
        distributing_df = distributing_df[distributing_df['hedged'] == False]

        #include_tickers = ['IBZL']
        #distributing_df = distributing_df[
        #    (distributing_df['ter'] <= 0.5) | (distributing_df['ticker'].isin(include_tickers))
        #]

        #Tickers to exclude (no london listing)
        remove_tickers = ['NADA','C001','C024','XDDA','ICHD','C030','XESD','XSMI','C005','XUCN']
        distributing_df = distributing_df[~distributing_df['ticker'].isin(remove_tickers)]



        # Merge benchmark data and compute beta for the entire distributing_df
        distributing_df = pd.merge(
            distributing_df,
            benchmark_df[['country', 'benchmark_ticker', 'benchmark_description', 'benchmark_2025_Return']],
            on='country', how='left'
        )
        distributing_df["beta"] = (
            distributing_df["2025"] / distributing_df["benchmark_2025_Return"]
        )

        hy_df = (distributing_df[distributing_df['beta'] >= 0.5].copy()
         .sort_values(by=['last_year_dividends','beta'], ascending=False)
         .drop_duplicates(subset=['region_category']))
        hy_df['intra_region_category'] = 'High Yield'

        benchmark_distributing_df = distributing_df.copy()[
            ~distributing_df['country'].isin(hy_df['country'])
        ]

        # Save intermediate benchmark data
        save_benchmark_etfs(benchmark_distributing_df, asset_class='equity', portfolio_year=2026)

        # Tolerance adjusted to 0.99 to allow valid ETFs to pass minor data feed discrepancies
        benchmark_distributing_df = benchmark_distributing_df[benchmark_distributing_df['beta'] >= 0.80]
        beta_df = (benchmark_distributing_df.copy()
                   .sort_values(by=['ter', 'beta'], ascending=[True, False])
                   .drop_duplicates(subset=['region_category']))
        beta_df['intra_region_category'] = 'Beta'

        main_df_equities = pd.concat([main_df_equities, hy_df, beta_df], ignore_index=True)

    except Exception as e:
        print(f'Error processing {csvl}: {e}')



# Fetch latest close price via DataProvider (yfinance, no API key)
def _get_price(ticker):
    try:
        return provider.get_latest_price(ticker)
    except Exception as e:
        print(f"Price error for {ticker}: {e}")
        return None, None

if not main_df_equities.empty:
    main_df_equities[['yday_close_price_date', 'yday_close_price']] = (
        main_df_equities['ticker'].apply(_get_price).to_list()
    )
else:
    main_df_equities['yday_close_price_date'] = []
    main_df_equities['yday_close_price'] = []
    print(
        "WARNING: main_df_equities is empty after screening. Likely causes: "
        "(1) no `data/raw/justetf_class-equity*.csv` files present, "
        "(2) InvestEngine API unreachable and Trading212 credentials missing, "
        "(3) every file raised an exception inside the per-file try/except. "
        "Re-run cell 4 (benchmarks) and cell 1 (imports), confirm InvestEngine "
        "is reachable, and check `data/raw/` is populated."
    )

save_screened_etfs(main_df_equities, 'equity', portfolio_year=2026)
print(f"Saved {len(main_df_equities)} equity ETFs")

# Per-region summary -- guarded against empty frames so a bad run surfaces
# cleanly instead of raising KeyError on the grouping columns.
if not main_df_equities.empty and {"region_category", "intra_region_category"}.issubset(main_df_equities.columns):
    display(
        main_df_equities
          .groupby(["region_category", "intra_region_category"])
          .size()
          .to_frame("count")
    )
else:
    print("No equity ETFs passed the platform/distribution filters -- nothing to summarise.")


Processing equity file for developed_americasanduk: justetf_class-equity_developed_americasanduk.csv
Processing equity file for developed_apac: justetf_class-equity_developed_apac.csv
Processing equity file for developed_emea: justetf_class-equity_developed_emea.csv
Processing equity file for emerging_americas: justetf_class-equity_emerging_americas.csv
Processing equity file for emerging_apacandemea: justetf_class-equity_emerging_apacandemea.csv
Saved 8 equity ETFs


count
region_category         intra_region_category       
Developed_APAC          High Yield                 1
Developed_AmericasandUK Beta                       1
                        High Yield                 1
Developed_EMEA          Beta                       1
                        High Yield                 1
Emerging_APACandEMEA    Beta                       1
                        High Yield                 1
Emerging_Americas       High Yield                 1

In [6]:
hy_df["beta"]

3    0.807849
Name: beta, dtype: float64

In [7]:
############# BONDS #############
main_df_bonds = pd.DataFrame()

# Manually curated bond ETF list
bonds_data = {
    'asset_class': ['bonds'] * 8,
    'region_category': [
        'Developed_AmericasandUK', 'Developed_AmericasandUK',
        'Developed_AmericasandUK', 'Developed_AmericasandUK',
        'Developed_EMEA', 'Developed_EMEA', 'Developed_EMEA',
        'Emerging_APACandEMEA',
    ],
    'intra_region_category': ['Govt', 'Corp', 'Govt', 'Corp', 'Govt', 'Corp', 'High yield', 'Corp'],
    'ticker': ['IGLT', 'SLXX', 'TRXG', 'UC81', 'PRIR', 'VECP', 'JNKE', 'EMCP'],
}
df_bonds_list = pd.DataFrame(bonds_data)

for filepath in sorted(DATA_RAW.glob("justetf_class-bonds_*.csv")):
    csvl = filepath.name
    try:
        csv_df = pd.read_csv(filepath)
        if csv_df.empty: continue
        
        # Filter raw data for our whitelist first
        for _, row in df_bonds_list.iterrows():
            if row['ticker'] in csv_df['ticker'].values:
                ticker = row['ticker']
                csv_df_ticker = csv_df[csv_df['ticker'] == ticker].copy()
                
                # [REVERTED] Platform check AFTER whitelist check
                csv_df_ticker['platform'] = check_platform(ticker, name=csv_df_ticker['name'].values[0])
                if pd.isna(csv_df_ticker['platform'].values[0]): continue
                
                csv_df_ticker['intra_region_category'] = row['intra_region_category']
                csv_df_ticker['region_category'] = row['region_category']
                csv_df_ticker['asset_class'] = row['asset_class']
                # Calculate beta relative to bond benchmark (SAAA)
                if bnd_benchmark_return != 0:
                    csv_df_ticker['beta'] = csv_df_ticker['2025'] / bnd_benchmark_return
                else:
                    csv_df_ticker['beta'] = 1.0
                main_df_bonds = pd.concat([main_df_bonds, csv_df_ticker], ignore_index=True)

    except Exception as e:
        print(f'Error processing {csvl}: {e}')

if not main_df_bonds.empty:
    main_df_bonds[['yday_close_price_date', 'yday_close_price']] = main_df_bonds['ticker'].apply(_get_price).to_list()
    save_screened_etfs(main_df_bonds, 'bonds', portfolio_year=2026)
    print(f"Saved {len(main_df_bonds)} bond ETFs")
    display(main_df_bonds[['ticker', 'name', 'intra_region_category', 'ter', 'platform']])


Saved 7 bond ETFs


,ticker,name,intra_region_category,ter,platform
0,IGLT,iShares Core UK Gilts UCITS ETF,Govt,0.07,InvestEngine
1,SLXX,iShares Core GBP Corporate Bond UCITS ETF,Corp,0.20,InvestEngine
2,TRXG,Invesco US Treasury Bond 7-10 Year UCITS ETF Dist,Govt,0.06,InvestEngine
3,UC81,UBS BBG US Liquid Corp 1-5 UCITS ETF USD dis,Corp,0.16,InvestEngine
4,EMCP,iShares J.P. Morgan USD EM Corporate Bond UCIT...,Corp,0.50,InvestEngine
5,PRIR,Amundi Prime Euro Government Bond UCITS ETF Dist,Govt,0.05,InvestEngine
6,VECP,Vanguard EUR Corporate Bond UCITS ETF (EUR) Di...,Corp,0.07,InvestEngine


In [8]:
############# PRECIOUS METALS & COMMODITIES #############

def _metal_type(name: str) -> str:
    n = str(name).lower()
    if 'silver' in n: return 'silver'
    if 'gold' in n: return 'gold'
    return 'other'

def _get_pm_beta(row):
    m = row['metal_type']
    if m in metal_benchmarks and not pd.isna(metal_benchmarks[m]):
        return (row.get('2025', 0) / 100.0) / metal_benchmarks[m]
    return 1.0

def _commodity_category(row) -> str:
    name = str(row['name']).lower()
    ticker = str(row['ticker'])
    if any(x in name for x in ['ex-', 'ex ', 'without']): return 'Broad'
    if ticker == 'NRGT': return 'Broad' 
    if any(x in name for x in ['energy', 'oil', 'petrol', 'fuel']): return 'Energy'
    if any(x in name for x in ['agri', 'wheat', 'corn', 'grain', 'softs']): return 'Agriculture'
    if any(x in name for x in ['metal', 'copper', 'alu', 'zinc', 'nickel']): return 'Industrial Metals'
    return 'Broad'

def _compute_cmd_beta(row):
    ticker = str(row['ticker'])
    cat = row['intra_region_category']
    if cat == 'Energy':
        bench = brent_ret if 'BRNT' in ticker else wti_ret
    elif cat == 'Agriculture':
        bench = agri_ret
    elif cat == 'Industrial Metals':
        bench = copper_ret
    else:
        return 1.0
    if bench and bench != 0:
        return (row.get('2025', 0) / 100.0) / bench
    return 1.0

# --- Precious Metals ---
pm_raw_path = DATA_RAW / 'justetf_class-preciousMetals_global.csv'
main_df_pm = pd.DataFrame()

if pm_raw_path.exists():
    pm_df = pd.read_csv(pm_raw_path)
    pm_df['asset_class'] = 'preciousMetals'
    pm_df['region_category'] = 'Global'
    pm_df['metal_type'] = pm_df['name'].apply(_metal_type)
    
    
    pm_filtered = pm_df[
        (pm_df['size'] > 100) & (pm_df['ter'] < 0.6) & (pm_df['hedged'] == False)
    ].copy()

    # Platform check after filtering
    pm_filtered['platform'] = pm_filtered.apply(lambda row: check_platform(row['ticker'], name=row['name']), axis=1)
    pm_filtered = pm_filtered[pm_filtered['platform'].notna()]

    #Tickers to exclude (no london listing, xgdp no price in AV, WSIL/RMAP/GLDW requires pre-clearance)
    remove_tickers = ['4GLD','XGDP','WSIL', 'RMAP','GLDW']
    pm_filtered = pm_filtered[~pm_filtered['ticker'].isin(remove_tickers)]

    print("Fetching dynamic precious metal benchmarks...")
    metal_benchmarks = {
        'gold': provider.get_benchmark_period_return('GOLD', start='2024-12-31', end='2025-12-31'),
        'silver': provider.get_benchmark_period_return('SILVER', start='2024-12-31', end='2025-12-31'),
    }
    pm_filtered['beta'] = pm_filtered.apply(_get_pm_beta, axis=1)
    pm_filtered = pm_filtered[pm_filtered['beta'] >= 0.78]

    main_df_pm = pm_filtered[pm_filtered['metal_type'].isin(['gold', 'silver'])].sort_values(['ter', 'size','beta']).groupby('metal_type').first().reset_index()
    main_df_pm['intra_region_category'] = main_df_pm['metal_type'].str.capitalize()
    main_df_pm[['yday_close_price_date', 'yday_close_price']] = main_df_pm['ticker'].apply(_get_price).to_list()
    save_screened_etfs(main_df_pm, 'preciousMetals', portfolio_year=2026)

# --- Commodities ---
print("Fetching dynamic sector benchmarks...")
wti_ret    = provider.get_benchmark_period_return('WTI', start='2024-12-31', end='2025-12-31')
brent_ret  = provider.get_benchmark_period_return('BRENT', start='2024-12-31', end='2025-12-31')
agri_ret   = provider.get_benchmark_period_return('WHEAT', start='2024-12-31', end='2025-12-31')
copper_ret = provider.get_benchmark_period_return('COPPER', start='2024-12-31', end='2025-12-31')
alum_ret   = provider.get_benchmark_period_return('ALUMINUM', start='2024-12-31', end='2025-12-31')

cmd_raw_path = DATA_RAW / 'justetf_class-commodities_global.csv'
main_df_cmd = pd.DataFrame()

if cmd_raw_path.exists():
    cmd_df = pd.read_csv(cmd_raw_path)
    cmd_df['asset_class'] = 'commodities'
    cmd_df['region_category'] = 'Global'
    cmd_df['intra_region_category'] = cmd_df.apply(_commodity_category, axis=1)
  
    remove_tickers = ['WEAP','AIGE']#no prices in alphavantage, AIGE is a broad energy etf
    cmd_df = cmd_df[~cmd_df['ticker'].isin(remove_tickers)]
    
  
    cmd_filtered = cmd_df[(cmd_df['size'] > 100) & (cmd_df['ter'] < 0.6) & (cmd_df['hedged'] == False)].copy()
    cmd_filtered['platform'] = cmd_filtered.apply(lambda row: check_platform(row['ticker'], name=row['name']), axis=1)
    cmd_filtered = cmd_filtered[cmd_filtered['platform'].notna()]

    

    cmd_filtered['beta'] = cmd_filtered.apply(_compute_cmd_beta, axis=1)
    cmd_filtered = cmd_filtered[cmd_filtered['beta'] >= 0.7]
    main_df_cmd = cmd_filtered[cmd_filtered['intra_region_category'].isin(['Energy', 'Agriculture', 'Industrial Metals'])].sort_values(['ter', 'size','beta']).groupby('intra_region_category').first().reset_index()
    main_df_cmd[['yday_close_price_date', 'yday_close_price']] = main_df_cmd['ticker'].apply(_get_price).to_list()
    save_screened_etfs(main_df_cmd, 'commodities', portfolio_year=2026)

# --- Summary ---
dfs = [df for df in [main_df_equities, main_df_bonds, main_df_pm, main_df_cmd] if not df.empty]
if dfs:
    summary_df = pd.concat(dfs, ignore_index=True)
    (
        summary_df[['asset_class', 'ticker', 'name', 'ter', 'beta', 'platform']]
        .rename(columns={
            'asset_class': 'Class',
            'ticker': 'Ticker',
            'name': 'Name',
            'ter': 'TER (%)',
            'beta': 'Beta',
            'platform': 'Platform',
        })
        .style.format({'TER (%)': '{:.2f}', 'Beta': '{:.2f}'})
        .hide(axis='index')
        .pipe(display)
    )

Fetching dynamic precious metal benchmarks...
Fetching commodity GOLD from AlphaVantage via GOLD_SILVER_HISTORY...
Fetching commodity SILVER from AlphaVantage via GOLD_SILVER_HISTORY...
Fetching dynamic sector benchmarks...
Fetching commodity WTI from AlphaVantage via WTI...
Fetching commodity BRENT from AlphaVantage via BRENT...
Fetching commodity WHEAT from AlphaVantage via WHEAT...
Fetching commodity COPPER from AlphaVantage via COPPER...
Fetching commodity ALUMINUM from AlphaVantage via ALUMINUM...


Class,Ticker,Name,TER (%),Beta,Platform
equity,IUKD,iShares UK Dividend UCITS ETF,0.40,1.46,InvestEngine
equity,XSTC,Xtrackers MSCI USA Information Technology UCITS ETF 1D,0.12,0.85,InvestEngine
equity,ISJP,iShares MSCI Japan Small Cap UCITS ETF (Dist),0.58,1.05,InvestEngine
equity,IMIB,iShares FTSE MIB UCITS ETF EUR (Dist),0.35,1.58,InvestEngine
equity,VGER,Vanguard Germany All Cap UCITS ETF (EUR) Distributing,0.07,0.97,InvestEngine
equity,IBZL,iShares MSCI Brazil UCITS ETF (Dist),0.74,1.22,InvestEngine
equity,HMCH,HSBC MSCI China UCITS ETF USD,0.28,0.81,InvestEngine
equity,HKOR,HSBC MSCI Korea Capped UCITS ETF USD,0.50,0.89,InvestEngine
bonds,IGLT,iShares Core UK Gilts UCITS ETF,0.07,1.68,InvestEngine
bonds,SLXX,iShares Core GBP Corporate Bond UCITS ETF,0.20,2.31,InvestEngine


The screened shortlist is saved to the database and carried forward to
[Portfolio Construction](03_portfolio_construction.ipynb).